# Differentiable Nonlinear-MPC for Autonomous Vehicle in JAX

![NLP Drift](Image/NLP_Drift.png)


Model Predictive Control (MPC) has arguably become a foundational framework and one of the most widely accepted modern control strategies due to its ability to explicitly incorporate system dynamics, and its ability to handle hard constraints on controls and states that arise in most applications. Particularly, it successfully and largely formalizes "prediction + constrained optimization + receding horizon feedback" in a way that is intuitive for practitioners and can incorporate real-world constraints directly. The philosophy of MPC is described as: "Predict future behaviour using a system model, given measurement or estimate of the current state of the system and a hypothetical future input trajectory or feedforward control policy"[1]. However, when systems operate at the limits of handling, such as automated drifting or aggressive obstacle avoidance, classical MPC approaches faces fundamental challenges[1]. 

Since, Automated drifting represents an extreme control regime in which the vehicle is deliberately driven into highly nonlinear and unstable operating regions characterized by large sideslip angles and tire saturation. In such conditions, the underlying assumptions of linear or mildly nonlinear models break down, and traditional MPC pipelines, typically based on repeated linearization and quadratic programming (QP), become computationally inefficient and potentially unreliable[11]. 

Parallel to this, recent developments in Nonlinear Model Predictive Control (NMPC) demonstrate that integrating motion planning, tracking, and stability into a single nonlinear optimization problem significantly improves performance in high-speed obstacle avoidance scenarios[11]. However, such approaches still rely on approximations to achieve real-time feasibility, often sacrificing model fidelity in highly coupled dynamic regimes. 

Another critical limitation arises from model uncertainty, i.e., real-world systems rarely conform to a single accurate model. The comparative study of multiple-model algorithms achieves strong robustness by fusing predictions from multiple candidate models, particularly under unknown or rapidly changing dynamics. This suggests NMPC becomes substantially harder when the controlled domain is intentionally operated beyond typical linear/near-equilibrium regimes. 

Automated drifting can be considered precisely such a domain as it requires maintaining high sideslip angles, exploiting nonlinear tire forces, and keeping rear tires saturated while still meeting trajectory objectives. Therefore, in general, drifting regime forces simultaneous engagement with four coupled difficulties. 1) Nonlinearities and strong coupling in which the tire force response is significantly nonlinear and coupled; maintaining drift depends on balancing front lateral forces (often within friction limits) against rear forces (beyond friction limits), with longitudinal slip at the rear playing a central role. 2) Unstable equilibria in which high sideslip drift equilibria can be unstable and the controller must regulate around equilibria rather than simply track a benign reference. 3) Tight actuator and rate constraints, which focuses that drifting requires aggressive steering and torque modulation, but real vehicles impose limits such as (e.g. steering torque/angle/rate/drivetrain dynamics). 4) Real-time computational constraints: MPC must solve an optimization problem every sample, more importantly, insisting on exact solutions is often impractical and sometimes undesirable due to computation delays and model mismatch, instead structure-exploiting approximate solutions can yield excellent closed-loop performance[9].


At the same time, advances in differentiable programming have introduced a paradigm shift in how optimization and computation are structured. Instead of treating derivatives as auxiliary computations, differentiable programming treats entire algorithms as composable, differentiable computational graphs. This enables efficient gradient computation, higher-order derivatives, and parallel execution through modern frameworks such as JAX, which provide composable transformations, automatic differentiation, JIT compilation via XLA, and vectorization that can be combined arbitrarily. The key opportunity emerging from these developments is to rethink NMPC as a compiled differentiable program instead of a sequence of optimization problems solved iteratively. Many components of NMPC such as trajectory simulation rollouts, cost accumulation, Jacobian/Hessian computations, Gauss–Newton approximations, KKT solves, and constraint evaluation are inherently differentiable and can be restructured into a unified computational graph. This restructuring allows [5]:

1). Efficient gradient computation via automatic differentiation.  
2). Parallel evaluation of multiple candidate trajectories  
3). Provide on-demand derivatives/HVPs with predictable complexity/memory trade-offs including checkpointing strategies.  

Therefore, building upon these insights, this project proposes a novel control framework, which integrates three key strategies that are physically meaningful.  
1) Differentiable Programming JAX-based NMPC: Recasting the NMPC problem as a fully differentiable program for efficient gradient-based optimization.  
2) Multi-model probabilistic fusion: Incorporating multiple dynamic models to handle uncertainty and model mismatch robustly to severe nonlinearities and saturation.  
3) Parallelized optimization architecture: Leveraging vectorization and just-in-time (JIT) compilation to achieve real-time performance based on differentiable programming and semi-decentralized coordination concepts.  



## Literature Review ## 

This literature review is organized around algorithmic structure that is basically  based on how optimization is actually performed and how is that structure with respect to nonlinearity, constraints, and real-time feasibility.  

A widely deployed NMPC implementation pattern is: 1) Choose a transcription (single shooting, multiple shooting, collocation), which then solves the resulting finite-dimensional NLP at each time step using SQP or interior-point methods, 2) apply the first control input and shift the horizon (receding horizon). And then MPC OCP is solved repeatedly and approximately; therefore, key errors stem from discretization and inexact optimization, and algorithmic choices must exploit OCP structure.  


Direct multiple shooting is also particularly important because it converts the continuous-time OCP into an NLP with explicit defect constraints linking shooting nodes, in which it resembles the discrete-time OCP form and has favorable numerical properties for unstable dynamics. This is directly relevant to drifting, where the dynamics around equilibria can be unstable and where convergence from poor guesses is a primary risk.  

SQP with Gauss–Newton structure is a classic route where the linearized dynamics/constraints approximate the Hessian (often Gauss–Newton), solve a QP, and iterate. The drifting NMPC cases use the ACADO toolkit with an SQP solver in real time.  

Therefore, across NMPC implementation, runtime is dominated by:  

1) Evaluating dynamics and constraints along the horizon  
2) Computing Jacobians and approximate Hessians  
3) Solving a structured linear system (QP/KKT) efficiently  

Therefore, differentiable programming literature [4] quantifies complexity/memory regimes, in which reverse-mode autodiff can compute gradients efficiently particularly for scalar outputs but requires storing intermediate computation, checkpointing trades recomputation for memory. These trade-offs matter directly for NMPC rollouts over horizons.  

Furthermore, drifting control is inherently sensitive to uncertainty. For instance, friction variations, tire parameter mismatch, and disturbances change equilibrium and can destabilize high sideslip operation. Among others, robust MPC literature provides two key tools:  
 
1) Tube MPC and constraint tightening: Tube MPC bounds deviations between nominal and actual trajectories using invariant sets, homothetic tube MPC introduces scaling variables $\alpha$ to reduce conservatism relative to rigid tubes. This is relevant to drifting because feasible sets near saturation can be narrow; naive tightening may eliminate feasible drift.  

2) Stochastic MPC and chance constraints: Stochastic uncertainty, especially multiplicative effects complicates prediction and constraint satisfaction. Stochastic MPC frameworks address   constraints via sampling/scenario approaches. For drifting, friction uncertainty behaves closer to multiplicative uncertainty on forces. A drifting NMPC pipeline should therefore integrate uncertainty either via robust tubes, stochastic models, or scenario-based validation.  

3) Distributed optimization and control: Modern MPC increasingly engages distributed optimization (ADMM, ALADIN) and scalable architectures. This becomes highly relevant if drifting control is extended to multi-vehicle scenarios, or if within-vehicle control is decomposed across actuators and subsystems (steering, powertrain, brake-based yaw moment, etc.) [6][7].  


The semi-decentralized multi-agent framework introduces a key abstraction with respect to probabilistic information flow via selector infrastructure that can toggle which observations/actions are stored in centralized vs. decentralized memories, defining mixed policies that interpolate between full decentralization and full centralization [8]. The RS-SDA* algorithm shows how this interpolation can preserve much of the value of centralized coordination while improving tractability and coping communication limitations. For NMPC, this motivates a semi-decentralized optimization architecture in which some decision variables (e.g. steering trajectory) are optimized centrally at high rate, others (e.g. wheel torque distribution, thermal constraints, actuator dynamics) are optimized locally with intermittent coupling updates and communication between modules is probabilistic or delayed, and the control algorithm is explicitly designed to remain stable under such information constraints. This concept aligns with real vehicle software architectures, where different ECUs operate at different rates and with constrained interfaces, which is exactly the difficulty encountered in the drifting NMPC experiments where steering torque availability through standard vehicle interfaces limited full automation [9].  

Therefore, from the literature, key gaps identified are:  
1) Non-convexity handling remains under-structured, where classical NMPC solvers often assume warm start + local convergence is enough. Drifting can invalidate this due to multiple equilibria and saturation-induced sensitivity.  
2) Derivative computation is treated as an implementation detail, not an algorithmic design axis; however, in the philosophy of differentiable programming, it argues the opposite, which is that program design and differentiation are central.  
3) Distributed/semi-decentralized control ideas are underused in vehicle NMPC, despite clear architectural relevance to real ECUs and actuator constraints.  
4) Validation is not integrated as a first-class driver of the solver design (e.g. scenario generation that targets solver failure modes, or differentiable sensi tivity analysis to expose brittle constraints).  




## Reconstructing the drifting NMPC baseline: 
















[1) REF: Kouvaritakis, Basil, and Mark Cannon. "Model predictive control." Switzerland: Springer International Publishing 38.13-56 (2016): 7. Chapter 1].  
[2) Mayne, David. "Nonlinear model predictive control: Challenges and opportunities." Nonlinear model predictive control (2000): 23-44.]  
[[3] Rawlings, James B., David Q. Mayne, and Moritz M. Diehl. "Model predictive control: theory, computation, and design." (No Title) (2020).]  
[[4] Blondel, Mathieu, and Vincent Roulet. "The elements of differentiable programming." arXiv preprint arXiv:2403.14606 (2024).]]  
[[5] Bradbury, James, et al. "JAX: Composable transformations of Python+ NumPy programs, version 0.3.13, 2018." Online at http://github.com/jax-ml/jax (accessed on October 31, 2025) (2018).]]  
[[6] Houska, Boris, and Yuning Jiang. "Distributed optimization and control with ALADIN." Recent Advances in Model Predictive Control: Theory, Algorithms, and Applications. Cham: Springer International Publishing, 2021. 135-163.]]  
[[7] Schulze Darup, Moritz, and Gerrit Book. "On closed-loop dynamics of ADMM-based MPC." Recent Advances in Model Predictive Control: Theory, Algorithms, and Applications. Cham: Springer International Publishing, 2021. 107-134.]  
[[8.] Al-Husseini, Mahdi, Mykel J. Kochenderfer, and Kyle H. Wray. "A Semi-Decentralized Approach to Multiagent Control." arXiv preprint arXiv:2603.11802 (2026).]  
[[9.]] Meijer, Stan, Alberto Bertipaglia, and Barys Shyrokau. "A nonlinear model predictive control for automated drifting with a standard passenger vehicle." 2024 IEEE International Conference on Advanced Intelligent Mechatronics (AIM). IEEE, 2024.  
[[10.]] Pitre, Ryan R., Vesselin P. Jilkov, and X. Rong Li. "A comparative study of multiple-model algorithms for maneuvering target tracking." Signal Processing, Sensor Fusion, and Target Recognition XIV. Vol. 5809. SPIE, 2005.

[[11.]] Chowdhri, Nishant, et al. "Integrated nonlinear model predictive control for automated driving." Control Engineering Practice 106 (2021): 104654.
[[12.]] Righetti, Giovanni, et al. "Drifting Maneuver Investigation via Phase Plane Analysis of Experimental Data." Advanced Vehicle Control Symposium. Cham: Springer Nature Switzerland, 2024.

## Proposed Timeline Plan (Which shall be improvised as we learn more and more, This is only a plan and raw IDEA :D )

1. **Complete the seminars session provided and suggested by the professor.**

   https://www.youtube.com/playlist?list=PLgnQpQtFTOGQo2Z_ogbonywTg8jxCI9pD
   
3. **USe the Seminar to build a broader understading of the research area.**
4. **Connect the perspectives and control and optimization.**
5. **Identify key challenges and solution approaches.**
6. **Refine the research focus of the project.**
7. **Study JAX libary to be used**

8. **Implement the proposed model by using JAX.**


In [2]:
import sys
import jax
import jax.numpy as jnp
print(jax.__version__)

0.9.0.1


In [4]:
!pip freeze

absl-py==2.4.0
aiofiles==25.1.0
anyio==4.12.1
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.1
async-lru==2.2.0
attrs==25.4.0
babel==2.18.0
beautifulsoup4==4.14.3
bleach==6.3.0
certifi==2026.2.25
cffi==2.0.0
charset-normalizer==3.4.5
chex==0.1.91
colorama==0.4.6
comm==0.2.3
dataclasses-json==0.6.7
debugpy==1.8.20
decorator==5.2.1
defusedxml==0.7.1
dm-haiku==0.0.16
etils==1.13.0
executing==2.2.1
fastjsonschema==2.21.2
flax==0.12.4
fqdn==1.5.1
fsspec==2026.2.0
h11==0.16.0
httpcore==1.0.9
httpx==0.28.1
humanize==4.15.0
idna==3.11
importlib_resources==6.5.2
ipykernel==7.2.0
ipython==9.11.0
ipython_pygments_lexers==1.1.1
isoduration==20.11.0
jax==0.9.0.1
jaxlib==0.9.0.1
jaxtyping==0.3.9
jedi==0.19.2
Jinja2==3.1.6
jmp==0.0.4
json5==0.13.0
jsonpointer==3.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-events==0.12.0
jupyter-lsp==2.3.0
jupyter_client==8.8.0
jupyter_core==5.9.1
jupyter_server==2.17.0
jupyter_server_terminals==0.5.4
jupyterlab==4.

In [6]:
os.listdir("Image")


['NLP_Drift.png']